# Phase 2 — Data Engineering

## 1. Building the Matches Table

### Objective

Build a clean and validated match-level table for the Premier League 2015/2016 season.

Each row represents one football match.

### Data Source

StatsBomb Open Data:

- Competition ID: `2`
- Season ID: `27`

### Data Contract

The `matches` table must contain:

- One row per match.
- A unique and non-null `match_id`.
- Valid home and away teams.
- Non-negative scores.
- Only matches from the selected competition and season.

In [746]:
from pathlib import Path

import pandas as pd
from statsbombpy import sb


COMPETITION_ID = 2
SEASON_ID = 27

# Locate the project root whether the notebook starts
# from the project folder or the notebooks folder
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bilal/Desktop/football-player-similarity


### Raw Data Storage

Before applying transformations, we store the complete source dataset in the raw data layer.

The raw layer:

- Preserves all columns returned by StatsBomb.
- Provides a reproducible copy of the source data.
- Allows the processed table to be rebuilt without downloading the data again.
- Must not contain manually transformed data.

In [747]:
matches_raw = sb.matches(
    competition_id=COMPETITION_ID,
    season_id=SEASON_ID
)

raw_matches_path = (
    RAW_DATA_DIR / "matches_raw.json"
)

matches_raw.to_json(
    raw_matches_path,
    orient="records",
    indent=2,
    force_ascii=False
)

print(f"Raw matches saved to: {raw_matches_path}")
print(f"Raw shape: {matches_raw.shape}")

Raw matches saved to: /Users/bilal/Desktop/football-player-similarity/data/raw/matches_raw.json
Raw shape: (380, 55)


/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [748]:


match_columns = [
    "match_id",
    "competition_id",
    "season_id",
    "match_date",
    "kick_off",
    "match_week",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "home_score",
    "away_score",
    "competition_stage",
    "stadium_id",
    "stadium",
    "referee_id",
    "referee",
]

matches = matches_raw[match_columns].copy()

# Convert the date from text to a proper date type
matches["match_date"] = pd.to_datetime(
    matches["match_date"]
)

print(f"Raw shape: {matches_raw.shape}")
print(f"Selected shape: {matches.shape}")

matches.head()

Raw shape: (380, 55)
Selected shape: (380, 17)


,match_id,competition_id,season_id,match_date,kick_off,match_week,home_team_id,home_team,away_team_id,away_team,home_score,away_score,competition_stage,stadium_id,stadium,referee_id,referee
0,3754217,2,27,2015-09-19,11:45:00.000,6,33,Chelsea,1,Arsenal,2,0,Regular Season,10,Stamford Bridge,6,Mike Dean
1,3754117,2,27,2015-12-13,12:30:00.000,16,59,Aston Villa,1,Arsenal,0,2,Regular Season,211,Villa Park,12,Kevin Friend
2,3754296,2,27,2015-12-21,19:00:00.000,17,1,Arsenal,36,Manchester City,2,1,Regular Season,3,Emirates Stadium,5,Andre Marriner
3,3753983,2,27,2015-10-31,14:00:00.000,11,26,Swansea City,1,Arsenal,0,3,Regular Season,45,Swansea.com Stadium,12,Kevin Friend
4,3754160,2,27,2015-12-05,14:00:00.000,15,1,Arsenal,41,Sunderland,3,1,Regular Season,3,Emirates Stadium,9,Robert Madley


### Data Quality Checks

Before storing the table, we verify its expected structure and integrity.

The validation checks:

- The season contains 380 matches.
- Every match ID is unique and non-null.
- Core match information is complete.
- A team cannot play against itself.
- Scores cannot be negative.
- The dataset contains 20 distinct teams.

In [749]:
core_columns = [
    "match_id",
    "match_date",
    "home_team_id",
    "home_team",
    "away_team_id",
    "away_team",
    "home_score",
    "away_score",
]

teams = set(matches["home_team"]) | set(matches["away_team"])

assert len(matches) == 380, "Unexpected number of matches"
assert matches["match_id"].is_unique, "match_id is not unique"
assert matches["match_id"].notna().all(), "Missing match_id detected"
assert matches[core_columns].notna().all().all(), "Missing core values"
assert (matches["home_team_id"] != matches["away_team_id"]).all()
assert (matches[["home_score", "away_score"]] >= 0).all().all()
assert matches["competition_id"].eq(COMPETITION_ID).all()
assert matches["season_id"].eq(SEASON_ID).all()
assert len(teams) == 20, "Unexpected number of teams"

print("All matches-table checks passed.")
print(f"Matches: {len(matches)}")
print(f"Unique match IDs: {matches['match_id'].nunique()}")
print(f"Teams: {len(teams)}")
print(
    "Date range:",
    matches["match_date"].min().date(),
    "to",
    matches["match_date"].max().date()
)

All matches-table checks passed.
Matches: 380
Unique match IDs: 380
Teams: 20
Date range: 2015-08-08 to 2016-05-17


## 2. Storing the Matches Table as Parquet

### Objective

Store the validated `matches` table in Parquet format.

### Why Parquet?

Parquet is a column-oriented data format designed for analytical workloads.

Compared with CSV, it:

- Preserves data types.
- Uses less disk space.
- Loads selected columns efficiently.
- Integrates directly with DuckDB.

In [750]:
matches_parquet_path = (
    PROCESSED_DATA_DIR / "matches.parquet"
)

matches.to_parquet(
    matches_parquet_path,
    index=False,
    engine="pyarrow"
)

file_size_kb = matches_parquet_path.stat().st_size / 1024

print(f"Parquet file created: {matches_parquet_path}")
print(f"File size: {file_size_kb:.2f} KB")

Parquet file created: /Users/bilal/Desktop/football-player-similarity/data/processed/matches.parquet
File size: 17.69 KB


### Parquet Round-Trip Validation

After writing the file, we read it back and compare it with the original DataFrame.

This verifies that:

- The file is readable.
- No rows or columns were lost.
- Column names were preserved.
- Values and data types were preserved.

In [751]:
matches_reloaded = pd.read_parquet(
    matches_parquet_path,
    engine="pyarrow"
)

pd.testing.assert_frame_equal(
    matches.reset_index(drop=True),
    matches_reloaded.reset_index(drop=True)
)

print("Parquet round-trip validation passed.")
print(f"Original shape: {matches.shape}")
print(f"Reloaded shape: {matches_reloaded.shape}")
print(
    "Unique reloaded match IDs:",
    matches_reloaded["match_id"].nunique()
)

Parquet round-trip validation passed.
Original shape: (380, 17)
Reloaded shape: (380, 17)
Unique reloaded match IDs: 380


## 3. Querying Parquet with DuckDB

### Objective

Use DuckDB to query the processed matches file directly with SQL.

DuckDB can read Parquet files without first loading them into a Pandas DataFrame.

In this first query, we select six columns, order the matches chronologically, and display the first five rows.

In [752]:
import duckdb

matches_preview = duckdb.sql(f"""
    SELECT
        match_id,
        match_date,
        home_team,
        away_team,
        home_score,
        away_score
    FROM read_parquet('{matches_parquet_path.as_posix()}')
    ORDER BY match_date
    LIMIT 5
""").df()

matches_preview

,match_id,match_date,home_team,away_team,home_score,away_score
0,3754097,2015-08-08,Manchester United,Tottenham Hotspur,1,0
1,3754237,2015-08-08,Leicester City,Sunderland,4,2
2,3754300,2015-08-08,Everton,Watford,2,2
3,3754138,2015-08-08,Norwich City,Crystal Palace,1,3
4,3754128,2015-08-08,AFC Bournemouth,Aston Villa,0,1


### Counting and Validating Matches with SQL

We use SQL aggregation functions to verify the number of rows, unique matches, and missing match identifiers.

This reproduces some of our Pandas quality checks using DuckDB.

In [753]:
matches_count = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT match_id) AS unique_matches,
        COUNT(*) - COUNT(match_id) AS missing_match_ids
    FROM read_parquet('{matches_parquet_path.as_posix()}')
""").df()

matches_count

,total_rows,unique_matches,missing_match_ids
0,380,380,0


## 4. Building the Events Dataset

### Objective

Build the event-level dataset for all matches in the season.

Each row will represent one event, such as:

- A pass
- A shot
- A carry
- A pressure
- A duel

Before processing all 380 matches, we test the extraction pipeline on one match.

This allows us to inspect the event schema and detect potential problems early.

In [754]:
first_match = (
    matches
    .sort_values(["match_date", "kick_off"])
    .iloc[0]
)

test_match_id = int(first_match["match_id"])

print(f"Test match ID: {test_match_id}")
print(
    f"Match: {first_match['home_team']} "
    f"vs {first_match['away_team']}"
)
print(f"Date: {first_match['match_date'].date()}")

Test match ID: 3754097
Match: Manchester United vs Tottenham Hotspur
Date: 2015-08-08


In [755]:
events_sample_raw = sb.events(
    match_id=test_match_id
)

print(f"Events shape: {events_sample_raw.shape}")
print(
    "Contains match_id column:",
    "match_id" in events_sample_raw.columns
)

events_sample_raw["type"].value_counts().head(10)

/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Events shape: (3780, 88)
Contains match_id column: True


type
Pass              1084
Ball Receipt*     1000
Carry              850
Pressure           373
Ball Recovery      102
Duel                74
Clearance           44
Dispossessed        37
Block               31
Foul Committed      26
Name: count, dtype: int64

### Core Event Schema

The raw event data contains many specialized columns.

We first define the core columns shared by the event types. These columns identify:

- The match and event.
- The event order and time.
- The team and player.
- The event type.
- The possession and spatial location.

Specialized passing, shooting, carrying, and defensive columns will be added separately.

In [756]:
CORE_EVENT_COLUMNS = [
    "match_id",
    "id",
    "index",
    "period",
    "timestamp",
    "minute",
    "second",
    "possession",
    "possession_team_id",
    "possession_team",
    "play_pattern",
    "team_id",
    "team",
    "player_id",
    "player",
    "position",
    "type",
    "location",
    "duration",
    "under_pressure",
    "counterpress",
]

In [757]:
events_core_sample = events_sample_raw.reindex(
    columns=CORE_EVENT_COLUMNS
).copy()

print(f"Raw shape: {events_sample_raw.shape}")
print(f"Core shape: {events_core_sample.shape}")

events_core_sample.head(10)

Raw shape: (3780, 88)
Core shape: (3780, 21)


,match_id,id,index,period,timestamp,minute,second,possession,possession_team_id,possession_team,...,team_id,team,player_id,player,position,type,location,duration,under_pressure,counterpress
0,3754097,70b5e25c-033c-43f1-adf5-ee871496669b,1,1,00:00:00.000,0,0,1,39,Manchester United,...,39,Manchester United,NaN,NaN,NaN,Starting XI,NaN,0.000000,NaN,NaN
1,3754097,58744932-5b80-42e0-8648-08eb48660154,2,1,00:00:00.000,0,0,1,39,Manchester United,...,38,Tottenham Hotspur,NaN,NaN,NaN,Starting XI,NaN,0.000000,NaN,NaN
2,3754097,d7bf0a57-243a-45e2-a597-fd214a5bd8ed,3,1,00:00:00.000,0,0,1,39,Manchester United,...,39,Manchester United,NaN,NaN,NaN,Half Start,NaN,0.000000,NaN,NaN
3,3754097,02309f9c-f8e1-4920-89ba-a9ea9ed84022,4,1,00:00:00.000,0,0,1,39,Manchester United,...,38,Tottenham Hotspur,NaN,NaN,NaN,Half Start,NaN,0.000000,NaN,NaN
4,3754097,e37edd92-100c-4565-ba69-41afab00e011,1973,2,00:00:00.000,45,0,91,39,Manchester United,...,39,Manchester United,NaN,NaN,NaN,Half Start,NaN,0.000000,NaN,NaN
5,3754097,1223d76f-6850-468a-ab48-8a52f88a3ef7,1974,2,00:00:00.000,45,0,91,39,Manchester United,...,38,Tottenham Hotspur,NaN,NaN,NaN,Half Start,NaN,0.000000,NaN,NaN
6,3754097,2f803d5d-ffb3-4353-892b-6c3f167a09aa,5,1,00:00:00.310,0,0,2,38,Tottenham Hotspur,...,38,Tottenham Hotspur,10955.0,Harry Kane,Center Forward,Pass,"[61.0, 40.1]",0.469881,NaN,NaN
7,3754097,23ebe4b5-40a5-4a93-9333-962fabe46672,7,1,00:00:00.780,0,0,2,38,Tottenham Hotspur,...,38,Tottenham Hotspur,3043.0,Christian Dannemann Eriksen,Center Attacking Midfield,Pass,"[62.7, 39.2]",0.347416,NaN,NaN
8,3754097,90d69842-089e-4028-8bca-98cee9f81f3d,10,1,00:00:01.578,0,1,2,38,Tottenham Hotspur,...,38,Tottenham Hotspur,8520.0,Nabil Bentaleb,Left Center Midfield,Pass,"[57.6, 39.2]",1.264742,NaN,NaN
9,3754097,05a0f86e-0f37-42e7-9c93-27491fe862e7,13,1,00:00:03.664,0,3,2,38,Tottenham Hotspur,...,38,Tottenham Hotspur,10956.0,Eric Dier,Right Center Midfield,Pass,"[42.8, 44.6]",4.249953,NaN,NaN


### Passing Event Schema

Passing columns describe the destination, distance, outcome, technique, and attacking value of each pass.

These variables will later be used to calculate player-level passing metrics such as:

- Passes attempted and completed.
- Pass completion percentage.
- Progressive passes.
- Crosses and switches.
- Through balls.
- Shot assists and goal assists.

In [758]:
PASS_EVENT_COLUMNS = [
    "pass_recipient_id",
    "pass_recipient",
    "pass_end_location",
    "pass_length",
    "pass_angle",
    "pass_height",
    "pass_body_part",
    "pass_type",
    "pass_outcome",
    "pass_technique",
    "pass_cross",
    "pass_cut_back",
    "pass_switch",
    "pass_through_ball",
    "pass_shot_assist",
    "pass_goal_assist",
    "pass_assisted_shot_id",
    "pass_aerial_won",
]

In [759]:
passes_sample = (
    events_sample_raw
    .loc[events_sample_raw["type"] == "Pass"]
    .reindex(
        columns=CORE_EVENT_COLUMNS + PASS_EVENT_COLUMNS
    )
    .copy()
)

print(f"Passes: {len(passes_sample)}")
print(f"Columns: {len(passes_sample.columns)}")

passes_sample[[
    "match_id",
    "minute",
    "player_id",
    "player",
    "location",
    "pass_end_location",
    "pass_length",
    "pass_outcome",
]].head(10)

Passes: 1084
Columns: 39


,match_id,minute,player_id,player,location,pass_end_location,pass_length,pass_outcome
6,3754097,0,10955.0,Harry Kane,"[61.0, 40.1]","[62.7, 39.5]",1.802776,NaN
7,3754097,0,3043.0,Christian Dannemann Eriksen,"[62.7, 39.2]","[58.1, 39.2]",4.600000,NaN
8,3754097,0,8520.0,Nabil Bentaleb,"[57.6, 39.2]","[43.6, 41.4]",14.171803,NaN
9,3754097,0,10956.0,Eric Dier,"[42.8, 44.6]","[97.7, 0.7]",70.293816,Out
10,3754097,0,4587.0,Matteo Darmian,"[26.4, 80.0]","[39.1, 66.2]",18.754467,NaN
11,3754097,0,3043.0,Christian Dannemann Eriksen,"[72.4, 15.3]","[56.8, 8.3]",17.098537,NaN
12,3754097,0,3077.0,Jan Vertonghen,"[50.9, 8.8]","[37.9, 29.5]",24.443610,NaN
13,3754097,0,20005.0,Toby Alderweireld,"[37.1, 42.2]","[42.0, 10.4]",32.175300,NaN
14,3754097,0,3077.0,Jan Vertonghen,"[42.8, 11.0]","[44.4, 25.2]",14.289857,NaN
15,3754097,0,8520.0,Nabil Bentaleb,"[41.7, 25.2]","[33.3, 49.5]",25.710892,NaN


### Creating the Pass Completion Indicator

StatsBomb records a successful pass with a missing `pass_outcome`.

We convert this convention into an explicit Boolean variable:

- `True`: The pass was completed.
- `False`: The pass was not completed.

This variable will later be aggregated by player and match.

In [760]:
passes_sample["pass_completed"] = (
    passes_sample["pass_outcome"].isna()
)

passes_sample[[
    "player",
    "pass_outcome",
    "pass_completed",
]].head(10)

,player,pass_outcome,pass_completed
6,Harry Kane,NaN,True
7,Christian Dannemann Eriksen,NaN,True
8,Nabil Bentaleb,NaN,True
9,Eric Dier,Out,False
10,Matteo Darmian,NaN,True
11,Christian Dannemann Eriksen,NaN,True
12,Jan Vertonghen,NaN,True
13,Toby Alderweireld,NaN,True
14,Jan Vertonghen,NaN,True
15,Nabil Bentaleb,NaN,True


In [761]:
passes_attempted = len(passes_sample)

passes_completed = int(
    passes_sample["pass_completed"].sum()
)

pass_completion_rate = (
    passes_completed
    / passes_attempted
    * 100
)

print(f"Passes attempted: {passes_attempted}")
print(f"Passes completed: {passes_completed}")
print(f"Completion rate: {pass_completion_rate:.2f}%")

Passes attempted: 1084
Passes completed: 860
Completion rate: 79.34%


### Shooting Event Schema

Shooting columns describe the quality, context, technique, and outcome of each shot.

These variables will later be used to calculate:

- Shots.
- Goals.
- Expected goals (xG).
- Non-penalty expected goals (npxG).
- Average xG per shot.

In [762]:
SHOT_EVENT_COLUMNS = [
    "shot_statsbomb_xg",
    "shot_outcome",
    "shot_type",
    "shot_body_part",
    "shot_technique",
    "shot_end_location",
    "shot_first_time",
    "shot_one_on_one",
    "shot_open_goal",
    "shot_deflected",
    "shot_key_pass_id",
]

In [763]:
shots_sample = (
    events_sample_raw
    .loc[events_sample_raw["type"] == "Shot"]
    .reindex(
        columns=CORE_EVENT_COLUMNS + SHOT_EVENT_COLUMNS
    )
    .copy()
)

print(f"Shots: {len(shots_sample)}")
print(f"Columns: {len(shots_sample.columns)}")

shots_sample[[
    "match_id",
    "minute",
    "player_id",
    "player",
    "location",
    "shot_statsbomb_xg",
    "shot_outcome",
    "shot_type",
]].head(10)

Shots: 18
Columns: 32


,match_id,minute,player_id,player,location,shot_statsbomb_xg,shot_outcome,shot_type
3611,3754097,4,3043.0,Christian Dannemann Eriksen,"[109.1, 32.5]",0.182832,Off T,Open Play
3612,3754097,11,10955.0,Harry Kane,"[107.1, 58.4]",0.012277,Blocked,Open Play
3613,3754097,11,20005.0,Toby Alderweireld,"[105.8, 39.3]",0.016993,Blocked,Open Play
3614,3754097,11,10955.0,Harry Kane,"[101.5, 42.1]",0.050492,Off T,Open Play
3615,3754097,17,3205.0,Kyle Walker,"[113.4, 49.6]",0.196066,Saved,Open Play
3616,3754097,27,3294.0,Juan Manuel Mata García,"[97.5, 43.7]",0.082456,Blocked,Free Kick
3617,3754097,27,2988.0,Memphis Depay,"[104.3, 30.8]",0.058164,Blocked,Open Play
3618,3754097,35,3294.0,Juan Manuel Mata García,"[103.5, 50.7]",0.040132,Blocked,Open Play
3619,3754097,35,3294.0,Juan Manuel Mata García,"[103.0, 39.3]",0.055361,Off T,Open Play
3620,3754097,40,2988.0,Memphis Depay,"[100.7, 28.4]",0.036374,Blocked,Open Play


### Creating Shooting Indicators

We create three derived shooting variables:

- `is_goal`: Indicates whether the shot resulted in a goal.
- `is_penalty`: Indicates whether the shot was a penalty.
- `non_penalty_xg`: Keeps the xG value for non-penalty shots and assigns zero to penalties.

These variables will later be aggregated by player and match.

In [764]:
shots_sample["is_goal"] = (
    shots_sample["shot_outcome"].eq("Goal")
)

shots_sample["is_penalty"] = (
    shots_sample["shot_type"].eq("Penalty")
)

shots_sample["non_penalty_xg"] = (
    shots_sample["shot_statsbomb_xg"]
    .where(
        ~shots_sample["is_penalty"],
        0.0
    )
)

shots_sample[[
    "player",
    "shot_outcome",
    "shot_type",
    "shot_statsbomb_xg",
    "is_goal",
    "is_penalty",
    "non_penalty_xg",
]].head(10)

,player,shot_outcome,shot_type,shot_statsbomb_xg,is_goal,is_penalty,non_penalty_xg
3611,Christian Dannemann Eriksen,Off T,Open Play,0.182832,False,False,0.182832
3612,Harry Kane,Blocked,Open Play,0.012277,False,False,0.012277
3613,Toby Alderweireld,Blocked,Open Play,0.016993,False,False,0.016993
3614,Harry Kane,Off T,Open Play,0.050492,False,False,0.050492
3615,Kyle Walker,Saved,Open Play,0.196066,False,False,0.196066
3616,Juan Manuel Mata García,Blocked,Free Kick,0.082456,False,False,0.082456
3617,Memphis Depay,Blocked,Open Play,0.058164,False,False,0.058164
3618,Juan Manuel Mata García,Blocked,Open Play,0.040132,False,False,0.040132
3619,Juan Manuel Mata García,Off T,Open Play,0.055361,False,False,0.055361
3620,Memphis Depay,Blocked,Open Play,0.036374,False,False,0.036374


In [765]:
shot_summary = pd.DataFrame({
    "shots": [len(shots_sample)],
    "goals": [int(shots_sample["is_goal"].sum())],
    "penalties": [int(shots_sample["is_penalty"].sum())],
    "total_xg": [shots_sample["shot_statsbomb_xg"].sum()],
    "non_penalty_xg": [shots_sample["non_penalty_xg"].sum()],
})

shot_summary.round(3)

,shots,goals,penalties,total_xg,non_penalty_xg
0,18,0,0,1.11,1.11


### Ball-Carrying and Dribbling Schema

Carries and dribbles describe two different forms of ball progression.

- A carry represents movement with the ball.
- A dribble represents an attempt to beat an opponent.

These events will later be used to calculate:

- Carries.
- Progressive carries.
- Dribbles attempted.
- Dribbles completed.
- Dribble completion percentage.

In [766]:
MOVEMENT_EVENT_COLUMNS = [
    "carry_end_location",
    "dribble_outcome",
    "dribble_overrun",
    "dribble_nutmeg",
]

In [767]:
movement_sample = (
    events_sample_raw
    .loc[
        events_sample_raw["type"].isin(
            ["Carry", "Dribble"]
        )
    ]
    .reindex(
        columns=CORE_EVENT_COLUMNS
        + MOVEMENT_EVENT_COLUMNS
    )
    .copy()
)

movement_sample["type"].value_counts()

type
Carry      850
Dribble     25
Name: count, dtype: int64

In [768]:
movement_sample[[
    "match_id",
    "minute",
    "player_id",
    "player",
    "type",
    "location",
    "carry_end_location",
    "dribble_outcome",
]].head(10)

,match_id,minute,player_id,player,type,location,carry_end_location,dribble_outcome
2090,3754097,0,8520.0,Nabil Bentaleb,Carry,"[58.1, 39.2]","[57.6, 39.2]",NaN
2091,3754097,0,10956.0,Eric Dier,Carry,"[43.6, 41.4]","[42.8, 44.6]",NaN
2092,3754097,0,3538.0,Wayne Mark Rooney,Carry,"[39.1, 66.2]","[40.2, 66.7]",NaN
2093,3754097,0,3043.0,Christian Dannemann Eriksen,Carry,"[78.3, 19.0]","[72.4, 15.3]",NaN
2094,3754097,0,3077.0,Jan Vertonghen,Carry,"[56.8, 8.3]","[50.9, 8.8]",NaN
2095,3754097,0,20005.0,Toby Alderweireld,Carry,"[37.9, 29.5]","[37.1, 42.2]",NaN
2096,3754097,0,3077.0,Jan Vertonghen,Carry,"[42.0, 10.4]","[42.8, 11.0]",NaN
2097,3754097,0,8520.0,Nabil Bentaleb,Carry,"[44.4, 25.2]","[41.7, 25.2]",NaN
2098,3754097,0,20005.0,Toby Alderweireld,Carry,"[33.3, 49.5]","[35.2, 56.7]",NaN
2099,3754097,0,3037.0,Mousa Sidi Yaya Dembélé,Carry,"[50.9, 69.1]","[42.0, 77.8]",NaN


### Creating the Dribble Completion Indicator

A dribble is completed when `dribble_outcome` equals `Complete`.

We first isolate dribble events to avoid incorrectly classifying carry events as failed dribbles.

In [769]:
dribbles_sample = (
    movement_sample
    .loc[movement_sample["type"] == "Dribble"]
    .copy()
)

dribbles_sample["dribble_completed"] = (
    dribbles_sample["dribble_outcome"].eq("Complete")
)

dribbles_sample[[
    "player",
    "minute",
    "dribble_outcome",
    "dribble_completed",
]].head(10)

,player,minute,dribble_outcome,dribble_completed
3555,Ben Davies,2,Complete,True
3556,Ashley Young,8,Incomplete,False
3557,Harry Kane,9,Incomplete,False
3558,Harry Kane,9,Complete,True
3559,Christian Dannemann Eriksen,16,Incomplete,False
3560,Nabil Bentaleb,17,Complete,True
3561,Ashley Young,19,Incomplete,False
3562,Ashley Young,27,Incomplete,False
3563,Harry Kane,38,Incomplete,False
3564,Wayne Mark Rooney,38,Complete,True


In [770]:
dribbles_attempted = len(dribbles_sample)

dribbles_completed = int(
    dribbles_sample["dribble_completed"].sum()
)

dribble_completion_rate = (
    dribbles_completed
    / dribbles_attempted
    * 100
)

print(f"Dribbles attempted: {dribbles_attempted}")
print(f"Dribbles completed: {dribbles_completed}")
print(
    f"Completion rate: "
    f"{dribble_completion_rate:.2f}%"
)

Dribbles attempted: 25
Dribbles completed: 13
Completion rate: 52.00%


### Defensive Event Schema

Defensive events describe how players recover possession, pressure opponents, contest the ball, and protect their goal.

These events will later be used to calculate:

- Pressures.
- Counterpressures.
- Tackles attempted and won.
- Interceptions.
- Ball recoveries.
- Clearances.
- Blocks.

In [771]:
DEFENSIVE_EVENT_COLUMNS = [
    "duel_type",
    "duel_outcome",
    "interception_outcome",
    "ball_recovery_recovery_failure",
    "ball_recovery_offensive",
    "clearance_aerial_won",
    "clearance_body_part",
    "block_deflection",
]

In [772]:
DEFENSIVE_EVENT_TYPES = [
    "Pressure",
    "Duel",
    "Interception",
    "Ball Recovery",
    "Clearance",
    "Block",
]

defensive_sample = (
    events_sample_raw
    .loc[
        events_sample_raw["type"].isin(
            DEFENSIVE_EVENT_TYPES
        )
    ]
    .reindex(
        columns=CORE_EVENT_COLUMNS
        + DEFENSIVE_EVENT_COLUMNS
    )
    .copy()
)

defensive_sample["type"].value_counts()

type
Pressure         373
Ball Recovery    102
Duel              74
Clearance         44
Block             31
Interception      15
Name: count, dtype: int64

In [773]:
defensive_sample[[
    "match_id",
    "minute",
    "player_id",
    "player",
    "type",
    "counterpress",
    "duel_type",
    "duel_outcome",
    "interception_outcome",
]].head(15)

,match_id,minute,player_id,player,type,counterpress,duel_type,duel_outcome,interception_outcome
2940,3754097,0,3086.0,Ben Davies,Pressure,NaN,NaN,NaN,NaN
2941,3754097,0,3612.0,Ashley Young,Pressure,NaN,NaN,NaN,NaN
2942,3754097,0,3382.0,Luke Shaw,Pressure,NaN,NaN,NaN,NaN
2943,3754097,0,4587.0,Matteo Darmian,Pressure,NaN,NaN,NaN,NaN
2944,3754097,1,4751.0,Michael Carrick,Pressure,NaN,NaN,NaN,NaN
2945,3754097,1,3600.0,Morgan Schneiderlin,Pressure,True,NaN,NaN,NaN
2946,3754097,1,3600.0,Morgan Schneiderlin,Pressure,True,NaN,NaN,NaN
2947,3754097,1,3538.0,Wayne Mark Rooney,Pressure,True,NaN,NaN,NaN
2948,3754097,1,2988.0,Memphis Depay,Pressure,NaN,NaN,NaN,NaN
2949,3754097,1,3311.0,Daley Blind,Pressure,NaN,NaN,NaN,NaN


### Building the Unified Events Schema

We combine the core and specialized column groups into one consistent event schema.

The resulting table preserves every event row while retaining only the variables required for the first version of the similarity engine.

In [774]:
EVENT_COLUMNS = list(dict.fromkeys(
    CORE_EVENT_COLUMNS
    + PASS_EVENT_COLUMNS
    + SHOT_EVENT_COLUMNS
    + MOVEMENT_EVENT_COLUMNS
    + DEFENSIVE_EVENT_COLUMNS
))

print(f"Selected event columns: {len(EVENT_COLUMNS)}")

Selected event columns: 62


In [775]:
events_sample = (
    events_sample_raw
    .reindex(columns=EVENT_COLUMNS)
    .copy()
)

print(f"Raw shape: {events_sample_raw.shape}")
print(f"Processed shape: {events_sample.shape}")

Raw shape: (3780, 88)
Processed shape: (3780, 62)


In [776]:
assert len(events_sample) == len(events_sample_raw)
assert events_sample["id"].notna().all()
assert events_sample["id"].is_unique
assert events_sample["match_id"].nunique() == 1
assert events_sample["match_id"].iloc[0] == test_match_id

print("Unified events validation passed.")
print(f"Events: {len(events_sample)}")
print(f"Unique event IDs: {events_sample['id'].nunique()}")
print(f"Match ID: {events_sample['match_id'].iloc[0]}")

Unified events validation passed.
Events: 3780
Unique event IDs: 3780
Match ID: 3754097


### Creating Event Indicators

We add explicit Boolean indicators to the unified events table.

These indicators will make player-match aggregation simpler and less error-prone.

Each indicator answers a specific question, such as:

- Is this event a pass?
- Is this event a completed pass?
- Is this event a shot or a goal?
- Is this event a carry, dribble, or defensive action?

In [777]:
events_sample["is_pass"] = (
    events_sample["type"].eq("Pass")
)

events_sample["pass_completed"] = (
    events_sample["is_pass"]
    & events_sample["pass_outcome"].isna()
)

events_sample["is_shot"] = (
    events_sample["type"].eq("Shot")
)

events_sample["is_goal"] = (
    events_sample["is_shot"]
    & events_sample["shot_outcome"].eq("Goal")
)

events_sample["is_penalty"] = (
    events_sample["is_shot"]
    & events_sample["shot_type"].eq("Penalty")
)

In [778]:
events_sample["xg"] = (
    events_sample["shot_statsbomb_xg"]
    .fillna(0.0)
)

events_sample["non_penalty_xg"] = (
    events_sample["xg"]
    .where(
        ~events_sample["is_penalty"],
        0.0
    )
)

In [779]:
events_sample["is_carry"] = (
    events_sample["type"].eq("Carry")
)

events_sample["is_dribble"] = (
    events_sample["type"].eq("Dribble")
)

events_sample["dribble_completed"] = (
    events_sample["is_dribble"]
    & events_sample["dribble_outcome"].eq("Complete")
)

events_sample["is_pressure"] = (
    events_sample["type"].eq("Pressure")
)

events_sample["is_counterpress"] = (
    events_sample["is_pressure"]
    & events_sample["counterpress"].eq(True)
)

events_sample["is_interception"] = (
    events_sample["type"].eq("Interception")
)

events_sample["is_ball_recovery"] = (
    events_sample["type"].eq("Ball Recovery")
)

events_sample["is_clearance"] = (
    events_sample["type"].eq("Clearance")
)

events_sample["is_block"] = (
    events_sample["type"].eq("Block")
)

In [780]:
indicator_columns = [
    "is_pass",
    "pass_completed",
    "is_shot",
    "is_goal",
    "is_penalty",
    "is_carry",
    "is_dribble",
    "dribble_completed",
    "is_pressure",
    "is_counterpress",
    "is_interception",
    "is_ball_recovery",
    "is_clearance",
    "is_block",
]

indicator_check = (
    events_sample[indicator_columns]
    .sum()
    .rename("event_count")
    .to_frame()
)

indicator_check

,event_count
is_pass,1084
pass_completed,860
is_shot,18
is_goal,0
is_penalty,0
is_carry,850
is_dribble,25
dribble_completed,13
is_pressure,373
is_counterpress,77


### Testing Event Storage in Parquet

Before processing all 380 matches, we test whether the unified event schema can be written to and read from Parquet.

Because this file contains only one test match, it belongs in the interim data layer.

The final season-level event dataset will later be stored in the processed layer.

In [781]:
events_sample_path = (
    INTERIM_DATA_DIR / "events_sample.parquet"
)

events_sample.to_parquet(
    events_sample_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

file_size_mb = (
    events_sample_path.stat().st_size
    / 1024
    / 1024
)

print(f"Sample saved to: {events_sample_path}")
print(f"File size: {file_size_mb:.2f} MB")

Sample saved to: /Users/bilal/Desktop/football-player-similarity/data/interim/events_sample.parquet
File size: 0.31 MB


In [782]:
events_sample_reloaded = pd.read_parquet(
    events_sample_path,
    engine="pyarrow"
)

print(f"Original shape: {events_sample.shape}")
print(f"Reloaded shape: {events_sample_reloaded.shape}")

Original shape: (3780, 78)
Reloaded shape: (3780, 78)


In [783]:
assert events_sample_reloaded.shape == events_sample.shape

assert list(events_sample_reloaded.columns) == list(
    events_sample.columns
)

assert events_sample_reloaded["id"].is_unique

assert (
    events_sample_reloaded["id"].nunique()
    == events_sample["id"].nunique()
)

reloaded_indicator_counts = (
    events_sample_reloaded[indicator_columns]
    .sum()
)

original_indicator_counts = (
    events_sample[indicator_columns]
    .sum()
)

pd.testing.assert_series_equal(
    reloaded_indicator_counts,
    original_indicator_counts
)

print("Events Parquet validation passed.")

Events Parquet validation passed.


### Creating a Reusable Event Transformation

We combine the validated event-selection and feature-engineering steps into one reusable function.

The function receives a raw StatsBomb event DataFrame and returns a consistent processed event DataFrame.

It will later be applied to every match in the season.

In [784]:
def prepare_events(events_raw):
    """
    Transform raw StatsBomb events into the processed
    event schema used by the similarity engine.
    """

    events = (
        events_raw
        .reindex(columns=EVENT_COLUMNS)
        .copy()
    )

    # Passing indicators
    events["is_pass"] = (
        events["type"].eq("Pass")
    )

    events["pass_completed"] = (
        events["is_pass"]
        & events["pass_outcome"].isna()
    )

    # Shooting indicators
    events["is_shot"] = (
        events["type"].eq("Shot")
    )

    events["is_goal"] = (
        events["is_shot"]
        & events["shot_outcome"].eq("Goal")
    )

    events["is_penalty"] = (
        events["is_shot"]
        & events["shot_type"].eq("Penalty")
    )

    events["xg"] = (
        events["shot_statsbomb_xg"]
        .fillna(0.0)
    )

    events["non_penalty_xg"] = (
        events["xg"]
        .where(
            ~events["is_penalty"],
            0.0
        )
    )

    # Ball movement indicators
    events["is_carry"] = (
        events["type"].eq("Carry")
    )

    events["is_dribble"] = (
        events["type"].eq("Dribble")
    )

    events["dribble_completed"] = (
        events["is_dribble"]
        & events["dribble_outcome"].eq("Complete")
    )

    # Defensive indicators
    events["is_pressure"] = (
        events["type"].eq("Pressure")
    )

    events["is_counterpress"] = (
        events["is_pressure"]
        & events["counterpress"].eq(True)
    )

    events["is_interception"] = (
        events["type"].eq("Interception")
    )

    events["is_ball_recovery"] = (
        events["type"].eq("Ball Recovery")
    )

    events["is_clearance"] = (
        events["type"].eq("Clearance")
    )

    events["is_block"] = (
        events["type"].eq("Block")
    )

    return events

In [785]:
events_sample_from_function = prepare_events(
    events_sample_raw
)

print(
    "Function output shape:",
    events_sample_from_function.shape
)

Function output shape: (3780, 78)


In [786]:
assert (
    events_sample_from_function.shape
    == events_sample.shape
)

assert list(events_sample_from_function.columns) == list(
    events_sample.columns
)

function_indicator_counts = (
    events_sample_from_function[indicator_columns]
    .sum()
)

pd.testing.assert_series_equal(
    function_indicator_counts,
    original_indicator_counts
)

print("Reusable event transformation validated.")

Reusable event transformation validated.


# TEST

In [787]:
import sys

project_root_string = str(
    PROJECT_ROOT.resolve()
)

if project_root_string not in sys.path:
    sys.path.insert(
        0,
        project_root_string
    )

print(sys.path[0])

/Users/bilal/Desktop/football-player-similarity


In [788]:
from src.events import (
    EVENT_COLUMNS as MODULE_EVENT_COLUMNS,
    prepare_events as prepare_events_from_module,
)

module_events_sample = prepare_events_from_module(
    events_sample_raw
)

assert MODULE_EVENT_COLUMNS == EVENT_COLUMNS

assert (
    module_events_sample.shape
    == events_sample.shape
)

module_indicator_counts = (
    module_events_sample[indicator_columns]
    .sum()
)

pd.testing.assert_series_equal(
    module_indicator_counts,
    original_indicator_counts
)

print("src.events module validated.")

src.events module validated.


## 5. Validating the Complete Events Dataset

The extraction pipeline produced one raw JSON file and one processed Parquet file per match.

We now use DuckDB to query all Parquet files as one logical dataset and verify:

- The number of files.
- The number of matches.
- The total number of events.
- The uniqueness of event identifiers.
- The absence of missing event identifiers.

In [789]:
RAW_EVENTS_DIR = (
    RAW_DATA_DIR / "events"
)

PROCESSED_EVENTS_DIR = (
    PROCESSED_DATA_DIR / "events"
)

raw_event_files = list(
    RAW_EVENTS_DIR.glob("*.json")
)

processed_event_files = list(
    PROCESSED_EVENTS_DIR.glob("*.parquet")
)

print(f"Raw JSON files: {len(raw_event_files)}")
print(
    f"Processed Parquet files: "
    f"{len(processed_event_files)}"
)

assert len(raw_event_files) == 380
assert len(processed_event_files) == 380

print("Event file-count validation passed.")

Raw JSON files: 380
Processed Parquet files: 380
Event file-count validation passed.


In [790]:
events_parquet_glob = (
    PROCESSED_EVENTS_DIR / "*.parquet"
).as_posix()

events_validation = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT match_id) AS total_matches,
        COUNT(DISTINCT id) AS unique_events,
        COUNT(*) - COUNT(id) AS missing_event_ids
    FROM read_parquet(
        '{events_parquet_glob}',
        union_by_name = true
    )
""").df()

events_validation

,total_events,total_matches,unique_events,missing_event_ids
0,1313773,380,1313773,0


In [791]:
validation_row = events_validation.iloc[0]

assert validation_row["total_events"] == 1_313_773
assert validation_row["total_matches"] == 380
assert (
    validation_row["unique_events"]
    == validation_row["total_events"]
)
assert validation_row["missing_event_ids"] == 0

print("Complete events dataset validation passed.")

Complete events dataset validation passed.


## 6. Exploring Match Lineups

### Objective

Retrieve player lineup information required to build the `player_match` table.

Lineup data provides:

- Players registered for each team.
- Starting and substitute status.
- Position intervals.
- Substitution times.
- Final playing position information.

These variables are required to calculate minutes played and per-90 metrics.

In [792]:
lineups_sample_raw = sb.lineups(
    match_id=test_match_id
)

print(type(lineups_sample_raw))
print(lineups_sample_raw.keys())

<class 'dict'>
dict_keys(['Manchester United', 'Tottenham Hotspur'])


/Users/bilal/Desktop/football-player-similarity/.venv/lib/python3.14/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [793]:
lineup_frames = []

for team_name, team_lineup in (
    lineups_sample_raw.items()
):
    team_lineup = team_lineup.copy()

    team_lineup["match_id"] = (
        test_match_id
    )

    team_lineup["team"] = (
        team_name
    )

    lineup_frames.append(
        team_lineup
    )

lineups_sample = pd.concat(
    lineup_frames,
    ignore_index=True,
)

print(f"Lineup shape: {lineups_sample.shape}")

lineups_sample[[
    "match_id",
    "team",
    "player_id",
    "player_name",
    "jersey_number",
    "positions",
]].head(10)

Lineup shape: (36, 9)


,match_id,team,player_id,player_name,jersey_number,positions
0,3754097,Manchester United,2988,Memphis Depay,7,"[{'position_id': 19, 'position': 'Center Attac..."
1,3754097,Manchester United,3058,Javier Hernández Balcázar,14,[]
2,3754097,Manchester United,3267,Luis Antonio Valencia Mosquera,25,"[{'position_id': 2, 'position': 'Right Back', ..."
3,3754097,Manchester United,3294,Juan Manuel Mata García,8,"[{'position_id': 17, 'position': 'Right Wing',..."
4,3754097,Manchester United,3311,Daley Blind,17,"[{'position_id': 5, 'position': 'Left Center B..."
5,3754097,Manchester United,3382,Luke Shaw,23,"[{'position_id': 6, 'position': 'Left Back', '..."
6,3754097,Manchester United,3538,Wayne Mark Rooney,10,"[{'position_id': 23, 'position': 'Center Forwa..."
7,3754097,Manchester United,3600,Morgan Schneiderlin,28,"[{'position_id': 15, 'position': 'Left Center ..."
8,3754097,Manchester United,3606,Ander Herrera Agüera,21,"[{'position_id': 19, 'position': 'Center Attac..."
9,3754097,Manchester United,3612,Ashley Young,18,"[{'position_id': 21, 'position': 'Left Wing', ..."


### Expanding Player Position Intervals

The `positions` column contains a list of playing intervals.

A player may have multiple intervals if:

- The player changed position.
- The player entered as a substitute.
- A tactical change occurred.

We expand these lists so that each row represents one player-position interval.

In [794]:
players_used_sample = (
    lineups_sample
    .loc[
        lineups_sample["positions"].str.len() > 0
    ]
    .copy()
)

print(
    "Players who entered the match:",
    len(players_used_sample)
)


Players who entered the match: 28


In [795]:
position_intervals_sample = (
    players_used_sample
    .explode(
        "positions",
        ignore_index=True,
    )
)

In [796]:
position_details = pd.json_normalize(
    position_intervals_sample["positions"]
)

position_intervals_sample = pd.concat(
    [
        position_intervals_sample
        .drop(columns="positions")
        .reset_index(drop=True),

        position_details
        .reset_index(drop=True),
    ],
    axis=1,
)

In [797]:
position_intervals_sample[[
    "match_id",
    "team",
    "player_id",
    "player_name",
    "position",
    "from",
    "to",
    "start_reason",
    "end_reason",
]].head(15)

,match_id,team,player_id,player_name,position,from,to,start_reason,end_reason
0,3754097,Manchester United,2988,Memphis Depay,Center Attacking Midfield,00:00,67:54,Starting XI,Substitution - Off (Tactical)
1,3754097,Manchester United,3267,Luis Antonio Valencia Mosquera,Right Back,80:02,NaN,Substitution - On (Tactical),Final Whistle
2,3754097,Manchester United,3294,Juan Manuel Mata García,Right Wing,00:00,NaN,Starting XI,Final Whistle
3,3754097,Manchester United,3311,Daley Blind,Left Center Back,00:00,NaN,Starting XI,Final Whistle
4,3754097,Manchester United,3382,Luke Shaw,Left Back,00:00,NaN,Starting XI,Final Whistle
5,3754097,Manchester United,3538,Wayne Mark Rooney,Center Forward,00:00,NaN,Starting XI,Final Whistle
6,3754097,Manchester United,3600,Morgan Schneiderlin,Left Center Midfield,00:00,NaN,Starting XI,Final Whistle
7,3754097,Manchester United,3606,Ander Herrera Agüera,Center Attacking Midfield,67:54,NaN,Substitution - On (Tactical),Final Whistle
8,3754097,Manchester United,3612,Ashley Young,Left Wing,00:00,NaN,Starting XI,Final Whistle
9,3754097,Manchester United,4587,Matteo Darmian,Right Back,00:00,80:02,Starting XI,Substitution - Off (Tactical)


In [798]:
match_event_seconds = (
    events_sample_raw["minute"] * 60
    + events_sample_raw["second"]
)

match_end_seconds = int(
    match_event_seconds.max()
)

match_end_minutes = (
    match_end_seconds / 60
)

print(
    f"Match duration: "
    f"{match_end_minutes:.2f} minutes"
)

Match duration: 94.37 minutes


### Calculating Position-Interval Durations

Position timestamps are stored as `minute:second` strings.

We convert them to seconds so that interval durations can be calculated numerically.

When the interval end is missing, the player remained on the pitch until the final whistle.

In [799]:
def clock_to_seconds(value):
    """Convert a minute:second value to seconds."""

    if pd.isna(value):
        return match_end_seconds

    minute_text, second_text = (
        str(value).split(":")
    )

    return (
        int(minute_text) * 60
        + int(second_text)
    )

In [800]:
position_intervals_sample["from_seconds"] = (
    position_intervals_sample["from"]
    .apply(clock_to_seconds)
)

position_intervals_sample["to_seconds"] = (
    position_intervals_sample["to"]
    .apply(clock_to_seconds)
)

position_intervals_sample["interval_seconds"] = (
    position_intervals_sample["to_seconds"]
    - position_intervals_sample["from_seconds"]
)

position_intervals_sample["interval_minutes"] = (
    position_intervals_sample["interval_seconds"]
    / 60
)

In [801]:
assert (
    position_intervals_sample["interval_seconds"]
    >= 0
).all()

position_intervals_sample[[
    "player_name",
    "position",
    "from",
    "to",
    "interval_minutes",
]].head(15)

,player_name,position,from,to,interval_minutes
0,Memphis Depay,Center Attacking Midfield,00:00,67:54,67.900000
1,Luis Antonio Valencia Mosquera,Right Back,80:02,NaN,14.333333
2,Juan Manuel Mata García,Right Wing,00:00,NaN,94.366667
3,Daley Blind,Left Center Back,00:00,NaN,94.366667
4,Luke Shaw,Left Back,00:00,NaN,94.366667
5,Wayne Mark Rooney,Center Forward,00:00,NaN,94.366667
6,Morgan Schneiderlin,Left Center Midfield,00:00,NaN,94.366667
7,Ander Herrera Agüera,Center Attacking Midfield,67:54,NaN,26.466667
8,Ashley Young,Left Wing,00:00,NaN,94.366667
9,Matteo Darmian,Right Back,00:00,80:02,80.033333


### Aggregating Minutes by Player

A player may have multiple position intervals during one match.

We aggregate these intervals to calculate:

- Total minutes played.
- Whether the player started the match.
- The player's primary position.

The primary position is defined as the position in which the player spent the most time.

In [802]:
player_minutes_sample = (
    position_intervals_sample
    .groupby(
        [
            "match_id",
            "team",
            "player_id",
            "player_name",
            "jersey_number",
        ],
        as_index=False,
    )
    .agg(
        minutes_played=(
            "interval_minutes",
            "sum",
        ),
        started=(
            "start_reason",
            lambda values: (
                values == "Starting XI"
            ).any(),
        ),
    )
)

In [803]:
primary_position_indices = (
    position_intervals_sample
    .groupby(
        ["match_id", "player_id"]
    )["interval_seconds"]
    .idxmax()
)

In [804]:
primary_positions_sample = (
    position_intervals_sample
    .loc[
        primary_position_indices,
        [
            "match_id",
            "player_id",
            "position_id",
            "position",
        ],
    ]
    .rename(
        columns={
            "position_id": "primary_position_id",
            "position": "primary_position",
        }
    )
)

In [805]:
player_match_base_sample = (
    player_minutes_sample
    .merge(
        primary_positions_sample,
        on=["match_id", "player_id"],
        how="left",
        validate="one_to_one",
    )
)

In [806]:
player_match_base_sample[
    [
        "match_id",
        "team",
        "player_id",
        "player_name",
        "started",
        "minutes_played",
        "primary_position",
    ]
].sort_values(
    ["team", "minutes_played"],
    ascending=[True, False],
).assign(
    minutes_played=lambda dataframe: (
        dataframe["minutes_played"].round(2)
    )
).head(30)

,match_id,team,player_id,player_name,started,minutes_played,primary_position
2,3754097,Manchester United,3294,Juan Manuel Mata García,True,94.37,Right Wing
3,3754097,Manchester United,3311,Daley Blind,True,94.37,Left Center Back
4,3754097,Manchester United,3382,Luke Shaw,True,94.37,Left Back
5,3754097,Manchester United,3538,Wayne Mark Rooney,True,94.37,Center Forward
6,3754097,Manchester United,3600,Morgan Schneiderlin,True,94.37,Left Center Midfield
8,3754097,Manchester United,3612,Ashley Young,True,94.37,Left Wing
11,3754097,Manchester United,4892,Sergio Germán Romero,True,94.37,Goalkeeper
12,3754097,Manchester United,10958,Chris Smalling,True,94.37,Right Center Back
9,3754097,Manchester United,4587,Matteo Darmian,True,80.03,Right Back
0,3754097,Manchester United,2988,Memphis Depay,True,67.90,Center Attacking Midfield


### Validating Player Minutes

For a match without red cards, each team should have 11 players on the pitch throughout the match.

Therefore:

total team minutes = 11 × match duration

We also verify that each team has exactly 11 starting players.

In [807]:
team_minutes_validation = (
    player_match_base_sample
    .groupby(
        ["match_id", "team"],
        as_index=False,
    )
    .agg(
        players_used=(
            "player_id",
            "nunique",
        ),
        starters=(
            "started",
            "sum",
        ),
        total_player_minutes=(
            "minutes_played",
            "sum",
        ),
    )
)

team_minutes_validation[
    "expected_player_minutes"
] = (
    match_end_minutes * 11
)

team_minutes_validation[
    "minutes_difference"
] = (
    team_minutes_validation[
        "total_player_minutes"
    ]
    - team_minutes_validation[
        "expected_player_minutes"
    ]
)

team_minutes_validation.round(2)

,match_id,team,players_used,starters,total_player_minutes,expected_player_minutes,minutes_difference
0,3754097,Manchester United,14,11,1038.03,1038.03,0.0
1,3754097,Tottenham Hotspur,14,11,1038.03,1038.03,0.0


In [808]:
assert (
    team_minutes_validation["starters"] == 11
).all()

assert (
    team_minutes_validation[
        "minutes_difference"
    ].abs() < 0.05
).all()

print("Player-minute validation passed.")

Player-minute validation passed.


# test


In [809]:
import importlib
import src.lineups as lineups_module

importlib.reload(lineups_module)

module_player_minutes_sample = (
    lineups_module.prepare_player_minutes(
        lineups_raw=lineups_sample_raw,
        match_id=test_match_id,
        match_end_seconds=match_end_seconds,
    )
)

print(
    "Module output shape:",
    module_player_minutes_sample.shape
)

Module output shape: (28, 9)


In [810]:
comparison_columns = [
    "match_id",
    "team",
    "player_id",
    "minutes_played",
    "started",
    "primary_position",
]

expected_player_minutes = (
    player_match_base_sample[
        comparison_columns
    ]
    .sort_values(
        ["team", "player_id"]
    )
    .reset_index(drop=True)
)

actual_player_minutes = (
    module_player_minutes_sample[
        comparison_columns
    ]
    .sort_values(
        ["team", "player_id"]
    )
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    actual_player_minutes,
    expected_player_minutes,
    check_dtype=False,
)

print("src.lineups module validated.")

src.lineups module validated.


## 7. Validating the Player-Minutes Dataset

We validate the complete player-minutes dataset before using it to build `player_match`.

The checks cover:

- File counts.
- Match coverage.
- Unique player-match combinations.
- Missing identifiers.
- Playing-time ranges.
- Starting-player counts.

In [811]:
RAW_LINEUPS_DIR = (
    RAW_DATA_DIR / "lineups"
)

PLAYER_MINUTES_DIR = (
    INTERIM_DATA_DIR / "player_minutes"
)

raw_lineup_files = list(
    RAW_LINEUPS_DIR.glob("*.json")
)

player_minutes_files = list(
    PLAYER_MINUTES_DIR.glob("*.parquet")
)

print(
    f"Raw lineup files: "
    f"{len(raw_lineup_files)}"
)

print(
    f"Player-minute files: "
    f"{len(player_minutes_files)}"
)

assert len(raw_lineup_files) == 380
assert len(player_minutes_files) == 380

print("Player-minute file counts validated.")

Raw lineup files: 380
Player-minute files: 380
Player-minute file counts validated.


In [812]:
player_minutes_glob = (
    PLAYER_MINUTES_DIR / "*.parquet"
).as_posix()

player_minutes_validation = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_player_matches,

        COUNT(DISTINCT match_id)
            AS total_matches,

        COUNT(
            DISTINCT (match_id, player_id)
        ) AS unique_player_matches,

        SUM(
            CASE
                WHEN player_id IS NULL THEN 1
                ELSE 0
            END
        ) AS missing_player_ids,

        SUM(
            CASE
                WHEN started THEN 1
                ELSE 0
            END
        ) AS total_starters,

        MIN(minutes_played)
            AS minimum_minutes,

        MAX(minutes_played)
            AS maximum_minutes

    FROM read_parquet(
        '{player_minutes_glob}',
        union_by_name = true
    )
""").df()

player_minutes_validation.round(2)

,total_player_matches,total_matches,unique_player_matches,missing_player_ids,total_starters,minimum_minutes,maximum_minutes
0,10469,380,10469,0.0,8360.0,0.37,101.18


In [813]:
import importlib

import src.lineups as lineups_module
import src.extract_lineups as extract_lineups_module

importlib.reload(lineups_module)
importlib.reload(extract_lineups_module)

<module 'src.extract_lineups' from '/Users/bilal/Desktop/football-player-similarity/src/extract_lineups.py'>

In [814]:
affected_match_id = 3754047

raw_path = (
    extract_lineups_module.RAW_LINEUPS_DIR
    / f"{affected_match_id}.json"
)

processed_path = (
    extract_lineups_module.PLAYER_MINUTES_DIR
    / f"player_minutes_{affected_match_id}.parquet"
)

lineups_raw = (
    extract_lineups_module.load_raw_lineups(
        raw_path
    )
)

match_end_seconds = (
    extract_lineups_module.get_match_end_seconds(
        affected_match_id
    )
)

corrected_player_minutes = (
    lineups_module.prepare_player_minutes(
        lineups_raw=lineups_raw,
        match_id=affected_match_id,
        match_end_seconds=match_end_seconds,
    )
)

corrected_player_minutes.to_parquet(
    processed_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

corrected_player_minutes.groupby(
    "team"
)["started"].sum()

team
Liverpool       11
Swansea City    11
Name: started, dtype: int64

In [815]:
starter_anomalies = duckdb.sql(f"""
    SELECT
        match_id,
        team,

        COUNT(*) AS players_used,

        SUM(
            CASE
                WHEN started THEN 1
                ELSE 0
            END
        ) AS starters

    FROM read_parquet(
        '{player_minutes_glob}',
        union_by_name = true
    )

    GROUP BY
        match_id,
        team

    HAVING
        SUM(
            CASE
                WHEN started THEN 1
                ELSE 0
            END
        ) <> 11

    ORDER BY
        match_id,
        team
""").df()

starter_anomalies

,match_id,team,players_used,starters


In [816]:
validation = player_minutes_validation.iloc[0]

assert validation["total_player_matches"] == 10469
assert validation["total_matches"] == 380
assert validation["unique_player_matches"] == 10469
assert validation["missing_player_ids"] == 0
assert validation["total_starters"] == 8360

assert validation["minimum_minutes"] > 0
assert starter_anomalies.empty

print(
    "Complete player-minutes dataset "
    "validation passed."
)

Complete player-minutes dataset validation passed.


## 8. Building the Player-Match Dataset

### 8.1 Aggregating Events by Player and Match

The event dataset contains one row per action. We aggregate these
actions to obtain one row per player and match.

Playing time will be joined in the next step.

In [817]:
events_glob = (
    PROCESSED_DATA_DIR
    / "events"
    / "*.parquet"
).as_posix()

event_player_match = duckdb.sql(f"""
    SELECT
        CAST(match_id AS BIGINT) AS match_id,
        CAST(team_id AS BIGINT) AS team_id,
        team,

        CAST(player_id AS BIGINT) AS player_id,
        player AS player_name,

        SUM(
            CASE WHEN is_pass THEN 1 ELSE 0 END
        ) AS passes_attempted,

        SUM(
            CASE WHEN pass_completed THEN 1 ELSE 0 END
        ) AS passes_completed,

        SUM(
            CASE WHEN is_shot THEN 1 ELSE 0 END
        ) AS shots,

        SUM(
            CASE WHEN is_goal THEN 1 ELSE 0 END
        ) AS goals,

        SUM(xg) AS total_xg,
        SUM(non_penalty_xg) AS non_penalty_xg,

        SUM(
            CASE WHEN is_carry THEN 1 ELSE 0 END
        ) AS carries,

        SUM(
            CASE WHEN is_dribble THEN 1 ELSE 0 END
        ) AS dribbles_attempted,

        SUM(
            CASE WHEN dribble_completed THEN 1 ELSE 0 END
        ) AS dribbles_completed,

        SUM(
            CASE WHEN is_pressure THEN 1 ELSE 0 END
        ) AS pressures,

        SUM(
            CASE WHEN is_counterpress THEN 1 ELSE 0 END
        ) AS counterpressures,

        SUM(
            CASE WHEN is_interception THEN 1 ELSE 0 END
        ) AS interceptions,

        SUM(
            CASE WHEN is_ball_recovery THEN 1 ELSE 0 END
        ) AS ball_recoveries,

        SUM(
            CASE WHEN is_clearance THEN 1 ELSE 0 END
        ) AS clearances,

        SUM(
            CASE WHEN is_block THEN 1 ELSE 0 END
        ) AS blocks

    FROM read_parquet(
        '{events_glob}',
        union_by_name = true
    )

    WHERE player_id IS NOT NULL

    GROUP BY
        match_id,
        team_id,
        team,
        player_id,
        player

    ORDER BY
        match_id,
        team,
        player_id
""").df()

In [818]:
key_columns = [
    "match_id",
    "player_id",
]

duplicate_count = (
    event_player_match
    .duplicated(key_columns)
    .sum()
)

assert duplicate_count == 0

assert (
    event_player_match[key_columns]
    .notna()
    .all()
    .all()
)

assert (
    event_player_match["passes_completed"]
    <= event_player_match["passes_attempted"]
).all()

assert (
    event_player_match["dribbles_completed"]
    <= event_player_match["dribbles_attempted"]
).all()

assert (
    event_player_match["goals"]
    <= event_player_match["shots"]
).all()

assert event_player_match["total_xg"].ge(0).all()
assert event_player_match["match_id"].nunique() == 380

print(
    "Event aggregation validation passed."
)
print(
    "Player-match rows with events:",
    len(event_player_match),
)
print(
    "Matches:",
    event_player_match["match_id"].nunique(),
)

event_player_match.head(10)

Event aggregation validation passed.
Player-match rows with events: 10450
Matches: 380


,match_id,team_id,team,player_id,player_name,passes_attempted,passes_completed,shots,goals,total_xg,non_penalty_xg,carries,dribbles_attempted,dribbles_completed,pressures,counterpressures,interceptions,ball_recoveries,clearances,blocks
0,3753972,28,AFC Bournemouth,3049,Matt Ritchie,50.0,37.0,2.0,0.0,0.119851,0.119851,39.0,0.0,0.0,14.0,5.0,2.0,7.0,4.0,5.0
1,3753972,28,AFC Bournemouth,3085,Glenn Murray,2.0,1.0,1.0,0.0,0.115622,0.115622,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3753972,28,AFC Bournemouth,3304,Harry Arter,69.0,57.0,1.0,0.0,0.021993,0.021993,60.0,2.0,0.0,21.0,5.0,0.0,11.0,1.0,1.0
3,3753972,28,AFC Bournemouth,3341,Steve Cook,52.0,45.0,4.0,0.0,0.242076,0.242076,39.0,0.0,0.0,5.0,2.0,0.0,7.0,8.0,2.0
4,3753972,28,AFC Bournemouth,3343,Dan Gosling,52.0,43.0,5.0,1.0,0.473784,0.473784,48.0,2.0,2.0,19.0,2.0,1.0,6.0,1.0,2.0
5,3753972,28,AFC Bournemouth,3344,Andrew Surman,57.0,51.0,0.0,0.0,0.000000,0.000000,42.0,2.0,1.0,8.0,1.0,1.0,8.0,6.0,2.0
6,3753972,28,AFC Bournemouth,3345,Charlie Daniels,61.0,52.0,0.0,0.0,0.000000,0.000000,40.0,1.0,1.0,3.0,0.0,0.0,5.0,6.0,1.0
7,3753972,28,AFC Bournemouth,3346,Joshua King,20.0,19.0,3.0,1.0,0.757376,0.757376,26.0,5.0,2.0,18.0,3.0,1.0,4.0,1.0,1.0
8,3753972,28,AFC Bournemouth,3463,Junior Stanislas,50.0,34.0,1.0,0.0,0.060181,0.060181,41.0,5.0,2.0,12.0,3.0,3.0,6.0,1.0,2.0
9,3753972,28,AFC Bournemouth,3608,Simon Francis,85.0,63.0,0.0,0.0,0.000000,0.000000,58.0,3.0,3.0,14.0,2.0,2.0,4.0,3.0,2.0


### 8.2 Joining Events with Playing Time

The player-minutes dataset is used as the main table so that every
player who participated in a match is preserved, including players
with no recorded event.

Event statistics and match metadata are then joined to this table.

In [819]:
player_minutes_glob = (
    INTERIM_DATA_DIR
    / "player_minutes"
    / "*.parquet"
).as_posix()

matches_path = (
    PROCESSED_DATA_DIR
    / "matches.parquet"
).as_posix()

player_match = duckdb.sql(f"""
    SELECT
        pm.match_id,
        m.competition_id,
        m.season_id,
        m.match_date,

        CASE
            WHEN pm.team = m.home_team
                THEN m.home_team_id
            WHEN pm.team = m.away_team
                THEN m.away_team_id
        END AS team_id,

        pm.team,
        pm.player_id,
        pm.player_name,
        pm.jersey_number,
        pm.started,
        pm.minutes_played,
        pm.primary_position_id,
        pm.primary_position,

        CAST(
            COALESCE(e.passes_attempted, 0)
            AS INTEGER
        ) AS passes_attempted,

        CAST(
            COALESCE(e.passes_completed, 0)
            AS INTEGER
        ) AS passes_completed,

        CAST(
            COALESCE(e.shots, 0)
            AS INTEGER
        ) AS shots,

        CAST(
            COALESCE(e.goals, 0)
            AS INTEGER
        ) AS goals,

        COALESCE(e.total_xg, 0.0)
            AS total_xg,

        COALESCE(e.non_penalty_xg, 0.0)
            AS non_penalty_xg,

        CAST(
            COALESCE(e.carries, 0)
            AS INTEGER
        ) AS carries,

        CAST(
            COALESCE(e.dribbles_attempted, 0)
            AS INTEGER
        ) AS dribbles_attempted,

        CAST(
            COALESCE(e.dribbles_completed, 0)
            AS INTEGER
        ) AS dribbles_completed,

        CAST(
            COALESCE(e.pressures, 0)
            AS INTEGER
        ) AS pressures,

        CAST(
            COALESCE(e.counterpressures, 0)
            AS INTEGER
        ) AS counterpressures,

        CAST(
            COALESCE(e.interceptions, 0)
            AS INTEGER
        ) AS interceptions,

        CAST(
            COALESCE(e.ball_recoveries, 0)
            AS INTEGER
        ) AS ball_recoveries,

        CAST(
            COALESCE(e.clearances, 0)
            AS INTEGER
        ) AS clearances,

        CAST(
            COALESCE(e.blocks, 0)
            AS INTEGER
        ) AS blocks

    FROM read_parquet(
        '{player_minutes_glob}',
        union_by_name = true
    ) AS pm

    INNER JOIN read_parquet(
        '{matches_path}'
    ) AS m
        ON pm.match_id = m.match_id

    LEFT JOIN event_player_match AS e
        ON pm.match_id = e.match_id
        AND pm.player_id = e.player_id

    ORDER BY
        pm.match_id,
        pm.team,
        pm.player_id
""").df()

In [820]:
player_match_key = [
    "match_id",
    "player_id",
]

assert len(player_match) == 10469

assert (
    player_match
    .duplicated(player_match_key)
    .sum()
    == 0
)

assert player_match["match_id"].nunique() == 380

assert (
    player_match[
        [
            "match_id",
            "team_id",
            "player_id",
            "minutes_played",
        ]
    ]
    .notna()
    .all()
    .all()
)

assert player_match["minutes_played"].gt(0).all()

assert (
    player_match["passes_completed"]
    <= player_match["passes_attempted"]
).all()

assert (
    player_match["dribbles_completed"]
    <= player_match["dribbles_attempted"]
).all()

print("Player-match join validation passed.")
print("Rows:", len(player_match))
print("Matches:", player_match["match_id"].nunique())
print("Players:", player_match["player_id"].nunique())

player_match.head(10)

Player-match join validation passed.
Rows: 10469
Matches: 380
Players: 550


,match_id,competition_id,season_id,match_date,team_id,team,player_id,player_name,jersey_number,started,...,non_penalty_xg,carries,dribbles_attempted,dribbles_completed,pressures,counterpressures,interceptions,ball_recoveries,clearances,blocks
0,3753972,2,27,2015-11-21,28,AFC Bournemouth,3049,Matt Ritchie,30,True,...,0.119851,39,0,0,14,5,2,7,4,5
1,3753972,2,27,2015-11-21,28,AFC Bournemouth,3085,Glenn Murray,27,False,...,0.115622,1,0,0,0,0,0,0,0,0
2,3753972,2,27,2015-11-21,28,AFC Bournemouth,3304,Harry Arter,8,True,...,0.021993,60,2,0,21,5,0,11,1,1
3,3753972,2,27,2015-11-21,28,AFC Bournemouth,3341,Steve Cook,3,True,...,0.242076,39,0,0,5,2,0,7,8,2
4,3753972,2,27,2015-11-21,28,AFC Bournemouth,3343,Dan Gosling,4,True,...,0.473784,48,2,2,19,2,1,6,1,2
5,3753972,2,27,2015-11-21,28,AFC Bournemouth,3344,Andrew Surman,6,True,...,0.000000,42,2,1,8,1,1,8,6,2
6,3753972,2,27,2015-11-21,28,AFC Bournemouth,3345,Charlie Daniels,11,True,...,0.000000,40,1,1,3,0,0,5,6,1
7,3753972,2,27,2015-11-21,28,AFC Bournemouth,3346,Joshua King,17,True,...,0.757376,26,5,2,18,3,1,4,1,1
8,3753972,2,27,2015-11-21,28,AFC Bournemouth,3463,Junior Stanislas,19,True,...,0.060181,41,5,2,12,3,3,6,1,2
9,3753972,2,27,2015-11-21,28,AFC Bournemouth,3608,Simon Francis,2,True,...,0.000000,58,3,3,14,2,2,4,3,2


### 8.3 Storing the Player-Match Table

The validated player-match dataset is stored as a single Parquet file.

This table contains one row per player and match and will be the
foundation for the player-season dataset.

In [821]:
player_match_path = (
    PROCESSED_DATA_DIR
    / "player_match.parquet"
)

player_match.to_parquet(
    player_match_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

file_size_mb = (
    player_match_path.stat().st_size
    / 1024
    / 1024
)

print(
    "Player-match Parquet created:",
    player_match_path,
)
print(f"File size: {file_size_mb:.2f} MB")

Player-match Parquet created: /Users/bilal/Desktop/football-player-similarity/data/processed/player_match.parquet
File size: 0.29 MB


In [822]:
player_match_reloaded = pd.read_parquet(
    player_match_path,
    engine="pyarrow",
)

pd.testing.assert_frame_equal(
    player_match.reset_index(drop=True),
    player_match_reloaded.reset_index(drop=True),
)

assert player_match_reloaded.shape == (10469, 28)

assert (
    player_match_reloaded
    .duplicated(["match_id", "player_id"])
    .sum()
    == 0
)

assert (
    player_match_reloaded["match_id"]
    .nunique()
    == 380
)

print(
    "Player-match Parquet validation passed."
)
print(
    "Shape:",
    player_match_reloaded.shape,
)

Player-match Parquet validation passed.
Shape: (10469, 28)


## 9. Building the Player-Season Dataset

The player-match table is aggregated into one row per player and
season.

The table preserves playing time, the primary position, appearances,
starts, and total event statistics. Per-90 features will be calculated
during the feature-engineering phase.

In [823]:
player_season = duckdb.sql("""
    WITH season_totals AS (
        SELECT
            competition_id,
            season_id,
            player_id,

            ANY_VALUE(player_name)
                AS player_name,

            CAST(
                COUNT(*) AS INTEGER
            ) AS appearances,

            CAST(
                SUM(
                    CASE
                        WHEN started THEN 1
                        ELSE 0
                    END
                ) AS INTEGER
            ) AS starts,

            SUM(minutes_played)
                AS minutes_played,

            CAST(
                SUM(passes_attempted) AS INTEGER
            ) AS passes_attempted,

            CAST(
                SUM(passes_completed) AS INTEGER
            ) AS passes_completed,

            CAST(
                SUM(shots) AS INTEGER
            ) AS shots,

            CAST(
                SUM(goals) AS INTEGER
            ) AS goals,

            SUM(total_xg)
                AS total_xg,

            SUM(non_penalty_xg)
                AS non_penalty_xg,

            CAST(
                SUM(carries) AS INTEGER
            ) AS carries,

            CAST(
                SUM(dribbles_attempted) AS INTEGER
            ) AS dribbles_attempted,

            CAST(
                SUM(dribbles_completed) AS INTEGER
            ) AS dribbles_completed,

            CAST(
                SUM(pressures) AS INTEGER
            ) AS pressures,

            CAST(
                SUM(counterpressures) AS INTEGER
            ) AS counterpressures,

            CAST(
                SUM(interceptions) AS INTEGER
            ) AS interceptions,

            CAST(
                SUM(ball_recoveries) AS INTEGER
            ) AS ball_recoveries,

            CAST(
                SUM(clearances) AS INTEGER
            ) AS clearances,

            CAST(
                SUM(blocks) AS INTEGER
            ) AS blocks

        FROM player_match_reloaded

        GROUP BY
            competition_id,
            season_id,
            player_id
    ),

    position_totals AS (
        SELECT
            competition_id,
            season_id,
            player_id,
            primary_position_id,
            primary_position,

            SUM(minutes_played)
                AS position_minutes

        FROM player_match_reloaded

        GROUP BY
            competition_id,
            season_id,
            player_id,
            primary_position_id,
            primary_position
    ),

    ranked_positions AS (
        SELECT
            *,

            ROW_NUMBER() OVER (
                PARTITION BY
                    competition_id,
                    season_id,
                    player_id

                ORDER BY
                    position_minutes DESC,
                    primary_position
            ) AS position_rank

        FROM position_totals
    )

    SELECT
        totals.competition_id,
        totals.season_id,
        totals.player_id,
        totals.player_name,

        positions.primary_position_id,
        positions.primary_position,

        totals.appearances,
        totals.starts,
        totals.minutes_played,

        totals.passes_attempted,
        totals.passes_completed,
        totals.shots,
        totals.goals,
        totals.total_xg,
        totals.non_penalty_xg,
        totals.carries,
        totals.dribbles_attempted,
        totals.dribbles_completed,
        totals.pressures,
        totals.counterpressures,
        totals.interceptions,
        totals.ball_recoveries,
        totals.clearances,
        totals.blocks

    FROM season_totals AS totals

    LEFT JOIN ranked_positions AS positions
        ON totals.competition_id
            = positions.competition_id
        AND totals.season_id
            = positions.season_id
        AND totals.player_id
            = positions.player_id
        AND positions.position_rank = 1

    ORDER BY
        totals.minutes_played DESC,
        totals.player_name
""").df()

In [824]:
player_season_key = [
    "competition_id",
    "season_id",
    "player_id",
]

assert (
    player_season
    .duplicated(player_season_key)
    .sum()
    == 0
)

assert player_season["competition_id"].eq(2).all()
assert player_season["season_id"].eq(27).all()

assert (
    player_season[
        [
            "player_id",
            "player_name",
            "primary_position",
            "minutes_played",
        ]
    ]
    .notna()
    .all()
    .all()
)

assert player_season["minutes_played"].gt(0).all()

assert (
    player_season["appearances"]
    >= player_season["starts"]
).all()

additive_columns = [
    "minutes_played",
    "passes_attempted",
    "passes_completed",
    "shots",
    "goals",
    "total_xg",
    "non_penalty_xg",
    "carries",
    "dribbles_attempted",
    "dribbles_completed",
    "pressures",
    "counterpressures",
    "interceptions",
    "ball_recoveries",
    "clearances",
    "blocks",
]

for column in additive_columns:
    season_total = player_season[column].sum()
    match_total = player_match_reloaded[column].sum()

    assert abs(season_total - match_total) < 1e-6, (
        f"Aggregation mismatch: {column}"
    )

print("Player-season aggregation validation passed.")
print("Players:", len(player_season))
print("Shape:", player_season.shape)

player_season.head(10)

Player-season aggregation validation passed.
Players: 550
Shape: (550, 24)


,competition_id,season_id,player_id,player_name,primary_position_id,primary_position,appearances,starts,minutes_played,passes_attempted,...,non_penalty_xg,carries,dribbles_attempted,dribbles_completed,pressures,counterpressures,interceptions,ball_recoveries,clearances,blocks
0,2,27,3815,Kasper Schmeichel,1,Goalkeeper,38,38,3594.016667,1220,...,0.000000,484,1,1,6,1,0,213,8,1
1,2,27,3813,Wes Morgan,3,Right Center Back,38,38,3594.016667,849,...,2.948268,500,11,9,273,51,46,96,244,87
2,2,27,3608,Simon Francis,3,Right Center Back,38,38,3591.566667,2454,...,0.196907,1595,49,34,403,75,60,152,213,58
3,2,27,3344,Andrew Surman,10,Center Defensive Midfield,38,38,3590.933333,2309,...,0.762400,1544,26,17,539,99,56,218,82,65
4,2,27,20005,Toby Alderweireld,3,Right Center Back,38,38,3569.183333,2122,...,3.184085,1544,12,7,237,45,46,147,258,73
5,2,27,3522,Heurelho da Silva Gomes,1,Goalkeeper,38,38,3554.150000,1132,...,0.000000,371,0,0,7,1,0,222,2,1
6,2,27,3523,Craig Dawson,2,Right Back,38,38,3548.666667,1658,...,3.431996,778,30,15,443,63,44,122,189,105
7,2,27,4022,Darren Fletcher,13,Right Center Midfield,38,38,3533.583333,1516,...,1.839551,1218,67,39,547,126,29,281,39,54
8,2,27,3262,Łukasz Fabiański,1,Goalkeeper,37,37,3492.200000,1212,...,0.104197,550,0,0,5,0,0,165,2,1
9,2,27,10955,Harry Kane,23,Center Forward,38,38,3486.650000,1005,...,17.717991,1024,96,56,491,109,12,173,31,62


### 9.2 Storing the Player-Season Table

The validated player-season dataset is stored in Parquet format.

This table will be used as the input for feature engineering and the
player similarity model.

In [825]:
player_season_path = (
    PROCESSED_DATA_DIR
    / "player_season.parquet"
)

player_season.to_parquet(
    player_season_path,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

file_size_kb = (
    player_season_path.stat().st_size
    / 1024
)

print(
    "Player-season Parquet created:",
    player_season_path,
)
print(f"File size: {file_size_kb:.2f} KB")

Player-season Parquet created: /Users/bilal/Desktop/football-player-similarity/data/processed/player_season.parquet
File size: 57.04 KB


In [826]:
player_season_reloaded = pd.read_parquet(
    player_season_path,
    engine="pyarrow",
)

pd.testing.assert_frame_equal(
    player_season.reset_index(drop=True),
    player_season_reloaded.reset_index(drop=True),
)

assert player_season_reloaded.shape == (550, 24)

assert (
    player_season_reloaded
    .duplicated(
        [
            "competition_id",
            "season_id",
            "player_id",
        ]
    )
    .sum()
    == 0
)

assert (
    player_season_reloaded["minutes_played"]
    .gt(0)
    .all()
)

print(
    "Player-season Parquet validation passed."
)
print(
    "Shape:",
    player_season_reloaded.shape,
)

Player-season Parquet validation passed.
Shape: (550, 24)
